# 06: Measure your circuit on a quantum computer

**What you will learn:** ideal probabilities describe a model; real hardware returns finite-shot counts. A shot is one execution followed by measurement.

We will send a Bell circuit to Quafu and compare its counts with the ideal prediction.


## 1. Prepare the workshop

Run setup unchanged. Platform credentials are read automatically, never entered or displayed here. The saved result will also be used by notebook 06.


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. Build the circuit and check the ideal answer

Ideally only `00` and `11` appear, each with probability one half.


In [ ]:
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
ideal = fq.run(circuit)
print("Ideal probabilities:", ideal.state.abs().square().tolist())


## 3. Choose the hardware

Confirm today’s available target with your instructor. A shot is one execution and measurement. **Enabling the switch and running the submission cell creates a real QPU task with 1024 shots and uses your quota.**


In [ ]:
target = "quafu:Dongling"
shots = 1024
submit_to_hardware = False


### Prepare recovery files (run unchanged)

This folded cell protects saved tasks from accidental duplicate submissions. It checks the circuit/configuration and restores a matching receipt with `fq.restore_job()`. It never submits work. Storage and credentials are setup concerns, not quantum operations.


In [ ]:
from flagquantum.core.ir import ensure_circuit_ir

receipt_path = OUTPUTS / "quafu-bell.receipt.json"
guard_path = OUTPUTS / "quafu-bell.submission.json"
request = {
    "circuit_hash": ensure_circuit_ir(circuit).content_hash,
    "target": target,
    "shots": shots,
    "image": None,
    "project": None,
    "queue": None,
}
job = None
if submit_to_hardware:
    if not os.getenv("QUAFU_API_KEY"):
        raise RuntimeError("Configure Quafu access outside this notebook; nothing submitted.")
    if receipt_path.exists():
        saved = json.loads(guard_path.read_text())
        if any(saved.get(key) != value for key, value in request.items()):
            raise RuntimeError("Saved task belongs to another experiment. Keep its files and use a new receipt filename.")
        job = fq.restore_job(receipt_path)
    else:
        if (OUTPUTS / "quafu-bell.json").exists():
            raise RuntimeError("Existing hardware result found without a receipt. Keep it and reconcile the task before submitting.")
        try:
            with guard_path.open("x") as stream:
                json.dump(request, stream, indent=2)
        except FileExistsError as exc:
            raise RuntimeError("An earlier attempt has no receipt. Reconcile it on the platform; do not resubmit.") from exc


## 4. Submit through FlagQuantum

`fq.submit()` returns without waiting for execution. Save its receipt immediately with `job.save()` so the same task can be restored after a kernel restart. Rerunning this notebook restores the recorded task instead of creating another one.


In [ ]:
if submit_to_hardware and job is None:
    job = fq.submit(
        circuit, target=target, shots=shots, outputs=fq.counts(), compiler=None,
    )
    print("Task ID:", job.id)
    job.save(receipt_path)
elif job is not None:
    print("Restored task:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. Query progress

This is a single read-only query. `queued` is waiting, `running` is executing, and `succeeded` means the platform reports completion. Result delivery is a separate step and can still be delayed or unavailable.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. Wait briefly and read the result

`job.wait()` is a convenience for polling status and retrieving the result. A timeout leaves the task running. Rerun this cell later. Alternatively use `job.result()` once the status is `succeeded`. No local result is substituted for a remote failure.


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    counts = result.counts[0]
    assert sum(counts.values()) == shots
    print("Measured counts:", counts)
    print("Total shots:", sum(counts.values()))


### Save the result for notebook 06 (optional)

This folded export records the actual task ID and counts. It is not required to understand or use `fq.submit()`.


In [ ]:
if result is not None:
    record = {
        "source": "live", "status": "completed", "target": target,
        "shots": shots, "task_id": job.id, "counts": counts,
    }
    (OUTPUTS / "quafu-bell.json").write_text(json.dumps(record, indent=2))


## Take it home

The standard API is the same outside Jupyter. This short example creates a new real task each time it is deliberately invoked; configure access outside your source and save the receipt before waiting.

```python
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
job = fq.submit(circuit, target="quafu:Dongling", shots=1024, outputs=fq.counts())
job.save("bell.receipt.json")
print(job.id, job.status())
```

For a saved task: `job = fq.restore_job("bell.receipt.json")`. When complete, inspect `job.result().counts`.

**Explain the result:** why might `01` and `10` appear? Would more shots remove device noise? For result comparison, see the reference notebook 06_compare_results. See [Cloud job field notes](../CLOUD_JOBS.md) for setup and recovery details.
